In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn import preprocessing as preprocessing
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm, trange
from tensorboardX import SummaryWriter
from concurrent.futures import ThreadPoolExecutor
import time

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import figaspect

import scipy.ndimage as ndimage
from scipy.optimize import curve_fit
from scipy.stats import multivariate_normal
import scipy.stats as stats
from scipy.ndimage import gaussian_filter, zoom
from absl import app, flags
from IPython import get_ipython
from IPython.display import clear_output
from skimage import io, morphology, filters, util

import gc 
from absl import app, flags
import datetime
import os
import sys
import cv2
from IPython.display import clear_output
from IPython import get_ipython
import random
import pandas as pd
import torch.multiprocessing as mp

if __name__ == '__main__':
    mp.set_start_method('spawn', force=True) # Add this line

# LOCA-PRAM — training

Trains the LOCA-PRAM detector on synthetic tiles: real particle-free background FOVs (`../data/background/`) with PSF-shaped particles whose shape and amplitude statistics come from `../psf/psf_analysis.ipynb`.

Tiles are 256×256 native px, downsampled ×2 to 128×128 (`DOWNSCALE_FACTOR`). `tanh_scale` is fixed at 6.0. Checkpoints and TensorBoard logs are written to `runs/<timestamp>/` (`LOCA_PRAM_BEST.pth` = lowest eval loss).

In [ ]:
# =====================================================================
# DOWNSCALE + PSF + MARKER CONFIG
# =====================================================================
DOWNSCALE_FACTOR = 2

BASE_WINDOW_SIZE = (256, 256)
window_size = (BASE_WINDOW_SIZE[0] // DOWNSCALE_FACTOR,
               BASE_WINDOW_SIZE[1] // DOWNSCALE_FACTOR)

# PSF sigmas: direct measurements from new PSF analysis, divided by DOWNSCALE_FACTOR.
BASE_SIGMA_X_LOC,   BASE_SIGMA_X_SCALE = 9.79, 1.32
BASE_SIGMA_Y_LOC,   BASE_SIGMA_Y_SCALE = 7.88, 1.38
BASE_SIGMA_MIN     = 2.5
BASE_MARGIN        = 12

SIGMA_X_LOC   = BASE_SIGMA_X_LOC   / DOWNSCALE_FACTOR
SIGMA_X_SCALE = BASE_SIGMA_X_SCALE / DOWNSCALE_FACTOR
SIGMA_Y_LOC   = BASE_SIGMA_Y_LOC   / DOWNSCALE_FACTOR
SIGMA_Y_SCALE = BASE_SIGMA_Y_SCALE / DOWNSCALE_FACTOR
SIGMA_MIN     = BASE_SIGMA_MIN     / DOWNSCALE_FACTOR
MARGIN        = max(1, int(round(BASE_MARGIN / DOWNSCALE_FACTOR)))

# ---- PSF amplitude distribution ----
# Empirical PSF amplitudes (n=145 after outlier removal) modeled as a Beta
# bounded to [AMP_FLOOR, AMP_CEIL]: smooth, right-skewed, and (unlike the old
# Gaussian) strictly cannot emit sub-noise amplitudes. Source: amplitude_samples.npy,
# written by ../psf/psf_analysis.ipynb.
# NOTE: amplitude is an INTENSITY, not a length, so it is NOT divided by
# DOWNSCALE_FACTOR — average-pool downsampling preserves the peak of a
# well-sampled Gaussian.
import numpy as np
from scipy import stats as _stats
_amp = np.load('amplitude_samples.npy')
AMP_FLOOR  = 12.0
AMP_CEIL   = float(_amp.max() + 0.03 * (_amp.max() - _amp.min()))
AMP_BETA_A, AMP_BETA_B, _, _ = _stats.beta.fit(
    _amp, floc=AMP_FLOOR, fscale=AMP_CEIL - AMP_FLOOR)

def sample_amplitude(size=None):
    """Particle amplitude(s) drawn from the fitted Beta (native intensity units)."""
    return _stats.beta.rvs(AMP_BETA_A, AMP_BETA_B, loc=AMP_FLOOR,
                           scale=AMP_CEIL - AMP_FLOOR, size=size)

# Marker / banned-region config.
MARKER_THRESHOLD = 15
BUFFER_SIGMA_MULT = 5.0
BUFFER_PX = int(round(BUFFER_SIGMA_MULT * max(SIGMA_X_LOC, SIGMA_Y_LOC)))


def compute_forbidden_mask(tile, buffer_px=None, threshold=None):
    if buffer_px is None:
        buffer_px = BUFFER_PX
    if threshold is None:
        threshold = MARKER_THRESHOLD
    marker = tile <= threshold
    if buffer_px <= 0 or not marker.any():
        return marker
    from scipy.ndimage import distance_transform_edt
    dist = distance_transform_edt(~marker)
    return dist <= buffer_px


def downsample_image(img, factor=None):
    if factor is None:
        factor = DOWNSCALE_FACTOR
    if factor == 1:
        return img
    import cv2
    h, w = img.shape[:2]
    return cv2.resize(img, (w // factor, h // factor), interpolation=cv2.INTER_AREA)


GAUSSIAN_BLUR_SIGMA = 0.0

def smooth_image(img, sigma=None):
    if sigma is None:
        sigma = GAUSSIAN_BLUR_SIGMA
    if sigma <= 0:
        return img
    from scipy.ndimage import gaussian_filter
    return gaussian_filter(img, sigma=sigma)


print(f"DOWNSCALE_FACTOR = {DOWNSCALE_FACTOR}")
print(f"window_size      = {window_size}")
print(f"sigma_x ~ N({SIGMA_X_LOC:.2f}, {SIGMA_X_SCALE:.2f})")
print(f"sigma_y ~ N({SIGMA_Y_LOC:.2f}, {SIGMA_Y_SCALE:.2f})")
print(f"amplitude ~ Beta(a={AMP_BETA_A:.2f}, b={AMP_BETA_B:.2f}) on [{AMP_FLOOR:.1f}, {AMP_CEIL:.1f}]")
print(f"placement margin = {MARGIN}, marker buffer = {BUFFER_PX} px ({BUFFER_SIGMA_MULT:.1f}*sigma)")
print(f"gaussian blur    = {GAUSSIAN_BLUR_SIGMA} px")

## Prepare Background Tiles

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

def standardize_image(image):
    """
    Standardize the pixel values of a 16-bit image so that the resulting image
    has a mean of 0 and a standard deviation of 1.

    Parameters:
    - image: A NumPy array representing the 16-bit input image.

    Returns:
    - A NumPy array of the standardized image.
    """
    image = image.astype(np.float32) / 255.
    mean = np.mean(image)
    std = np.std(image)
    standardized_image = (image - mean) / std
    print(f"mean: {mean}, std: {std}")
    return standardized_image

def shadow_remove(img):     
    """
    (Explanation needed)

    Parameters:
    - image: A NumPy array representing the 16-bit input image.

    Returns:
    - A NumPy array of the shadow removed image.
    """   
    dilated_img = morphology.dilation(img, morphology.square(11))
    bg_img = filters.median(dilated_img, morphology.disk(31))
    shadowremov = img - bg_img
    return shadowremov

def save_image(image):
    print(image.dtype)
    # Before saving, rescale and convert the processed image back to 16-bit
    final_img_16bit = rescale_to_16bit(image)
    cv2.imwrite('background_subtracted.png', final_img_16bit)

def rescale_to_16bit(image_float, target_range=(0, 65535)):
    """
    Rescales a floating-point image to the specified 16-bit range.

    Parameters:
    - image_float: A NumPy array of the image in floating-point format.
    - target_range: A tuple specifying the desired output range (min, max).

    Returns:
    - A NumPy array of the image rescaled to the 16-bit range.
    """
    min_val = np.min(image_float)
    max_val = np.max(image_float)
    scale = (target_range[1] - target_range[0]) / (max_val - min_val)
    image_16bit = (image_float - min_val) * scale + target_range[0]
    return image_16bit.astype(np.uint16)

def visualize_image(images, labels, backgrounds, image_index=0, figsize=(2000, 800)):
    """
    Visualize the images. 
    - image:            Number of training images generated. 
                        (num_imgs, 1, img_width, img_height)
    - background:       Previously generated "bg_tiles" is used. 
                        (num_imgs, 1, img_width, img_height)
    - labels:           One-hot representation of the center of the PSFs. 
                        (num_imgs, 1, img_width, img_height)
    - labels_density:   When "labels" are the one-hot representation, "label_density" has pixel value of 100 followed by smoothing with gaussian filter 
                        (num_imgs, 1, img_width, img_height)
    - image_index:      For sanity check, a single image from the training set is displayed. 
                        (int)
    - figsize:          Size of the figure. 
                        ex) (1000, 1000)
                        
    Returns:
    - None
    """   

    if labels.ndim == 3:
        labels = labels.unsqueeze(0)
        backgrounds = backgrounds.unsqueeze(0)
    if images.ndim == 3:
        images = images.unsqueeze(0)
    
    print(f"image index: {image_index}")
    num_imgs, img_channel, img_size_x, img_size_y = images.shape
    px = 1/plt.rcParams['figure.dpi'] 
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(figsize[0]*px, figsize[1]*px))

    for i, ax in enumerate(axes.flatten()):
        D = np.squeeze(images[image_index, ...] )
        if not i % 4:
            GT = np.squeeze(labels[image_index, 2, ...] > 0)
            BG = np.squeeze(backgrounds[image_index, 0, ...])
            a = ax.imshow(D, origin='lower', cmap='gray')
            ax.set_title('Input Image')
        if i%4 == 1:
            a = ax.imshow(BG, origin='lower', cmap='gray')
            ax.set_title('Input Background')
        if i%4 == 2:
            a = ax.imshow(GT, origin='lower', cmap='gray')
            ax.set_title('Ground Truth Center')
            
        fig.colorbar(a, ax=ax, fraction=0.046)
        ax.set_xlim([0, img_size_x])
        ax.set_ylim([img_size_y, 0])

    plt.tight_layout()
    plt.show()
    
def prepare_background_tiles(bg_image_path, window_size):
    """
    Load a synthetic background image and tile it.
    
    Parameters:
    - bg_image_path: Path to the synthetic background image
    - window_size: Tuple (height, width) for each tile
    
    Returns:
    - bg_tiles: NumPy array of shape (num_tiles, height, width)
    """
    # Load image
    bg_image = np.asarray(Image.open(bg_image_path).convert('L')).astype(np.float32)
    if DOWNSCALE_FACTOR != 1:
        bg_image = downsample_image(bg_image)
        print(f"Downsampled background by {DOWNSCALE_FACTOR}x: {bg_image.shape}")
    print(f"Loaded background: {bg_image.shape}, range: [{bg_image.min():.1f}, {bg_image.max():.1f}]")
    
    # Normalize to 0-1
    bg_image = (bg_image - bg_image.min()) / (bg_image.max() - bg_image.min())
    
    bg_height, bg_width = bg_image.shape
    bg_tiles = []
    
    # Tile the image
    for i in range(int(bg_height / window_size[0])):
        for j in range(int(bg_width / window_size[1])):
            bg_tile = bg_image[i*window_size[0] : i*window_size[0] + window_size[0], 
                               j*window_size[1] : j*window_size[1] + window_size[1]]
            bg_tiles.append(bg_tile)
    
    bg_tiles = np.asarray(bg_tiles)
    print(f"Generated {len(bg_tiles)} tiles of size {window_size}")
    
    # Visualization
    index = 0
    fig, subplots = plt.subplots(1, 2, figsize=(16, 6))
    subplots[0].set_title("Synthetic Background")
    subplot_0 = subplots[0].imshow(bg_image, cmap='gray')
    rect = patches.Rectangle((window_size[1] * (index % int(bg_width / window_size[1])), 
                               window_size[0] * (index // int(bg_width / window_size[1]))), 
                              window_size[1], window_size[0], linewidth=2, linestyle='--', 
                              edgecolor='r', facecolor='none')
    subplots[0].add_patch(rect)
    fig.colorbar(subplot_0, ax=subplots[0], fraction=0.03)
    subplot_1 = subplots[1].imshow(bg_tiles[index].squeeze(), cmap='gray')
    subplots[1].set_title(f"Tile #{index}")
    fig.colorbar(subplot_1, ax=subplots[1], fraction=0.05)
    plt.tight_layout()
    plt.show()
    
    return bg_tiles

if __name__ == '__main__':
    # NOTE: original single-image loader disabled; the real loader is
    # in the next code cell (multi-FOV from ../data/background/).
    pass

## Authentic Image for Testing
### Sanity Check

In [ ]:
# =====================================================================
# Multi-FOV background loader
# ---------------------------------------------------------------------
# Walks BG_FOLDER, splits FOVs into train / eval by TRAIN_FOV_FRAC,
# takes the RED channel of each JPG, sub-tiles each FOV at window_size,
# and drops tiles with too little placeable area. Produces two pools:
# bg_tiles_train and bg_tiles_eval (held-out FOVs). bg_tiles is an alias
# of the train pool used by the visualization cells.
# =====================================================================
import glob
import os

BG_FOLDER = '../data/background/'
TRAIN_FOV_FRAC = 0.9     # fraction of FOVs used for training; rest held out
SPLIT_SEED = 42
# Min fraction of pixels that are placeable AFTER marker dilation + edge margin
# (a heavily-marked tile can have 0% placeable area at 128x128).
MIN_PLACEABLE_FRAC = 0.05


def load_fov_red(path):
    """Open a JPG FOV and return the red channel as float32 (0-255 range)."""
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    arr = arr.astype(np.float32)
    if DOWNSCALE_FACTOR != 1:
        arr = downsample_image(arr)
    return arr


def _placeable_frac(tile):
    """Fraction of pixels that pass marker dilation + edge margin — i.e. that
    generate_simulation_image could actually place a particle on."""
    forbidden = compute_forbidden_mask(tile)
    if MARGIN > 0:
        forbidden[:MARGIN, :] = True
        forbidden[-MARGIN:, :] = True
        forbidden[:, :MARGIN] = True
        forbidden[:, -MARGIN:] = True
    return float((~forbidden).mean())


def tile_fovs(fov_paths, window_size, label=""):
    tiles = []
    n_dropped = 0
    for p in fov_paths:
        img = load_fov_red(p)
        h, w = img.shape
        for i in range(h // window_size[0]):
            for j in range(w // window_size[1]):
                tile = img[i*window_size[0]:(i+1)*window_size[0],
                           j*window_size[1]:(j+1)*window_size[1]]
                if _placeable_frac(tile) < MIN_PLACEABLE_FRAC:
                    n_dropped += 1
                    continue
                # Blur applied AFTER the placeable filter (which needs raw
                # values for the marker threshold). No-op if sigma=0.
                tile = smooth_image(tile)
                tiles.append(tile)
    arr = np.asarray(tiles)
    print(f"  {label}: {len(fov_paths)} FOVs -> {len(arr)} tiles "
          f"(dropped {n_dropped} with placeable_frac < {MIN_PLACEABLE_FRAC:.0%})")
    return arr


all_fovs = sorted(glob.glob(os.path.join(BG_FOLDER, '*.jpg')))
assert len(all_fovs) >= 2, \
    f"need at least 2 FOVs in {BG_FOLDER} (train + eval), found {len(all_fovs)}"
n_train = max(1, min(len(all_fovs) - 1, int(round(len(all_fovs) * TRAIN_FOV_FRAC))))

rng = np.random.RandomState(SPLIT_SEED)
perm = rng.permutation(len(all_fovs))
train_fovs = [all_fovs[i] for i in perm[:n_train]]
eval_fovs  = [all_fovs[i] for i in perm[n_train:]]
print(f"Loaded {len(all_fovs)} FOVs from {BG_FOLDER}")
print(f"  train: {len(train_fovs)} FOVs")
print(f"  eval : {len(eval_fovs)} FOVs (held out)")

bg_tiles_train = tile_fovs(train_fovs, window_size, label="train")
bg_tiles_eval  = tile_fovs(eval_fovs,  window_size, label="eval ")

# Alias used by the visualization cells.
bg_tiles = bg_tiles_train

print(f"Tile value range (train): "
      f"[{bg_tiles_train.min():.1f}, {bg_tiles_train.max():.1f}]")

## Training Data Generation

In [ ]:
def generate_simulation_image(img_size, bg_tiles, num_gaus_low=1, num_gaus_high=40, labels_channel=3):
    """
    Generates a simulated image with synthetic PSF particles on a real
    background tile (sampled at random from bg_tiles).
    Amplitude drawn from the fitted Beta distribution (see config cell).
    """
    bg_idx = np.random.randint(0, len(bg_tiles))
    gauss_base = bg_tiles[bg_idx].copy()
    bg_clean   = bg_tiles[bg_idx]

    # ---- Forbidden mask: markers dilated by BUFFER_PX, plus edge margin ----
    forbidden = compute_forbidden_mask(bg_clean)
    if MARGIN > 0:
        forbidden[:MARGIN, :] = True
        forbidden[-MARGIN:, :] = True
        forbidden[:, :MARGIN] = True
        forbidden[:, -MARGIN:] = True

    placeable_ys, placeable_xs = np.where(~forbidden)
    active_frac = placeable_ys.size / forbidden.size

    x_vec = np.arange(img_size[1])
    y_vec = np.arange(img_size[0])

    labels = np.zeros((labels_channel, img_size[0], img_size[1]))

    # ---- Particle count: 15% empty tiles, otherwise density-aware ----
    EMPTY_TILE_PROB = 0.15   # forced-empty fraction for negative supervision
    if np.random.random() < EMPTY_TILE_PROB:
        n_to_place = 0
    else:
        nominal_count = np.random.randint(num_gaus_low, num_gaus_high)
        scaled = nominal_count * active_frac
        n_floor = int(scaled)
        n_to_place = n_floor + (1 if np.random.random() < (scaled - n_floor) else 0)

    if placeable_ys.size == 0:
        n_to_place = 0

    # Min-separation rejection sampling + random PSF rotation.
    MIN_SEPARATION = 5.0   # native model px (matches paper's setup)
    used_locations = []
    Xg, Yg = np.meshgrid(x_vec, y_vec, indexing='xy')

    # ---- Per-tile domain-gap augmentations ----
    # Defocus: symmetric widening, integrated intensity preserved.
    # Motion blur: rank-1 covariance term along a per-tile drift axis.
    # Both drawn once per tile (FOV-wide physical effects).
    if np.random.random() < 0.55:              # 55% of tiles stay sharp
        defocus = 1.0
    else:                                       # 45% get real defocus
        defocus = np.random.uniform(1.3, 1.8)
    motion_theta  = np.random.uniform(0.0, 2.0 * np.pi)
    motion_length = np.random.uniform(0.0, 1.5)          # px
    sigma_motion  = motion_length / 2.35                  # FWHM → σ
    _cm, _sm = np.cos(motion_theta), np.sin(motion_theta)
    Sigma_motion = sigma_motion ** 2 * np.array([
        [_cm * _cm, _cm * _sm],
        [_cm * _sm, _sm * _sm],
    ])
    for _ in range(n_to_place):
        # 1. Sample center from placeable pixels (rejection-sampled for min separation)
        placed = False
        for _attempt in range(50):
            k = np.random.randint(0, placeable_ys.size)
            iy_base = placeable_ys[k]
            ix_base = placeable_xs[k]
            x_c = ix_base + np.random.uniform(0, 1)
            y_c = iy_base + np.random.uniform(0, 1)
            if all((x_c - ux) ** 2 + (y_c - uy) ** 2 >= MIN_SEPARATION ** 2
                   for ux, uy in used_locations):
                used_locations.append((x_c, y_c))
                placed = True
                break
        if not placed:
            continue   # too crowded — skip this particle

        # 2. Sample amplitude from fitted Beta
        amplitude = float(sample_amplitude())

        # 3. Sample sigma (PSF-measured + downscaled values), apply defocus
        raw_sigma_x = np.random.normal(loc=SIGMA_X_LOC, scale=SIGMA_X_SCALE)
        raw_sigma_y = np.random.normal(loc=SIGMA_Y_LOC, scale=SIGMA_Y_SCALE)
        sigma_x = max(SIGMA_MIN, raw_sigma_x) * defocus
        sigma_y = max(SIGMA_MIN, raw_sigma_y) * defocus
        amplitude_eff = amplitude / (defocus ** 2)   # preserve integrated intensity

        # 4. Random PSF rotation theta ∈ [-π/2, π/2), then combine with motion blur
        theta = np.random.uniform(-np.pi / 2, np.pi / 2)
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        R_psf = np.array([[cos_t, -sin_t], [sin_t, cos_t]])
        Sigma_psf   = R_psf @ np.diag([sigma_x ** 2, sigma_y ** 2]) @ R_psf.T
        Sigma_total = Sigma_psf + Sigma_motion
        Sigma_inv   = np.linalg.inv(Sigma_total)

        # 5. Render using full covariance (handles anisotropic motion axis)
        Xr = Xg - x_c
        Yr = Yg - y_c
        q = (Sigma_inv[0, 0] * Xr * Xr
             + 2.0 * Sigma_inv[0, 1] * Xr * Yr
             + Sigma_inv[1, 1] * Yr * Yr)
        particle = amplitude_eff * np.exp(-0.5 * q)

        gauss_base -= particle

        # 6. Update labels (one-hot at the integer center)
        ix, iy = int(x_c), int(y_c)
        if 0 <= ix < img_size[1] and 0 <= iy < img_size[0]:
            labels[0, iy, ix] = x_c - ix
            labels[1, iy, ix] = y_c - iy
            labels[2, iy, ix] = 1.0

    gauss_base = np.clip(gauss_base, 0, 65535)
    return gauss_base, labels, bg_clean

In [ ]:
tile = bg_tiles[0]
print(f"Tile mean: {tile.mean():.1f}")
print(f"Tile std:  {tile.std():.1f}")
print(f"Tile local std (32x32 patches):")
for y in range(0, 512, 128):
    for x in range(0, 512, 128):
        patch = tile[y:y+32, x:x+32]
        print(f"  ({y},{x}): std={patch.std():.2f}")
print(f"Particle amplitude: ~10")
print(f"Noise level: ~2.7")

### Asynchronous Dataset Loading

In [ ]:
class GaussianDataset(Dataset):
    def __init__(self, num_batches, bg_tiles, img_size=(512, 512), transform=None):
        self.transform = transform
        self.num_batches = num_batches
        self.bg_tiles = bg_tiles
        self.img_size = img_size
        self.labels_channel = 3
        
        # --- DYNAMIC DENSITY CALCULATION ---
        # Goal: Keep particle-to-image coverage ratio consistent with the original version (20% coverage)
        
        # 1. Reference (Original Setup)
        # 50 particles (radius ~3px) in 84x84
        ref_coverage_ratio = 0.20 
        
        # 2. New Setup (Current Config)
        new_window_area = self.img_size[0] * self.img_size[1]
        
        
        # PSF area shrank ~16x (sigma 4x smaller); bumped 4x to 0-23 so we
        # actually get overlapping particles for the model to learn that case.
        self.num_gaus_low = 0
        self.num_gaus_high = 40
        
        # print(f"Density Calculation for {img_size}:")
        # print(f"  - Particle Dimensions: ~{int(new_sigma_x*4)}x{int(new_sigma_y*4)} pixels")
        # print(f"  - Max Particles set to: {self.num_gaus_high}")

    def __len__(self):
        return self.num_batches

    def __getitem__(self, idx):
        # We call the generate function we defined in the previous cell
        image, labels, bg_image = generate_simulation_image(
            img_size=self.img_size, 
            bg_tiles=self.bg_tiles, 
            num_gaus_low=self.num_gaus_low, 
            num_gaus_high=self.num_gaus_high,
            labels_channel=self.labels_channel
        )
        
        # Standardize / Normalize -- ignore marker pixels only (match real inference).
        valid = bg_image > MARKER_THRESHOLD          # was: bg_image > MARKER_THRESHOLD
        if valid.any():
            m = image[valid].mean()
            s = image[valid].std()
        else:
            m = image.mean()
            s = image.std()
        if s == 0:
            s = 1
        image = (image - m) / s
        
        # Normalize bg_image with the SAME (m, s) as input so the bg head
        # supervises in the same normalized space.
        bg_image = (bg_image - m) / s

        # Convert to Tensor
        image = torch.from_numpy(image).float().unsqueeze(0) # (1, H, W)
        labels = torch.from_numpy(labels).float()            # (3, H, W)
        bg_image = torch.from_numpy(bg_image).float().unsqueeze(0)
        
        # Apply transforms if they exist
        if self.transform:
            image = self.transform(image)
            
        return image, labels, bg_image

# Asynchronous data loading function for batches
def async_load_batch(data_loader):
    """Load batches asynchronously using a thread pool."""
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [executor.submit(load_batch, batch) for batch in data_loader]
        for future in futures:
            yield future.result()

def load_batch(batch):
    """Simulate batch loading (you can customize this to fit your process)."""
    time.sleep(0.1)  # Simulate a delay (remove or adjust depending on your loading time)
    return batch

# Sanity Check: Visualizing asynchronously generated data
def sanity_check(images):
    plt.figure(figsize=(8, 8))
    # Loop over generated data for visualization and checking
    for i in range(images.shape[0]):
        if i < 4 :
            plt.subplot(2, 2, i + 1)
            plt.imshow(images[i].squeeze(), cmap='gray')
            plt.colorbar()
            plt.title(f"Generated Image {i+1}")
            plt.axis('off')
    plt.tight_layout()
    plt.show()

if __name__ == '__main__':
    # Generate a few images and show background vs background+particles side by side
    print("Visual comparison: Background vs Background+Particles")
    
    num_examples = 4
    fig, axes = plt.subplots(num_examples, 3, figsize=(18, 6 * num_examples))
    
    for i in range(num_examples):
        image, labels, bg_image = generate_simulation_image(
            img_size=window_size,
            bg_tiles=bg_tiles,
            num_gaus_low=0,
            num_gaus_high=40,
            labels_channel=3
        )
        
        diff = image - bg_image  # isolates particles + noise
        
        # A. Background only
        axes[i, 0].imshow(bg_image, cmap='gray')
        axes[i, 0].set_title(f'Background (range: {bg_image.min():.1f}–{bg_image.max():.1f})')
        fig.colorbar(axes[i, 0].images[0], ax=axes[i, 0], fraction=0.046)
        
        # B. Background + particles
        axes[i, 1].imshow(image, cmap='gray')
        num_particles = int(labels[2].sum())
        axes[i, 1].set_title(f'With {num_particles} particles (range: {image.min():.1f}–{image.max():.1f})')
        fig.colorbar(axes[i, 1].images[0], ax=axes[i, 1], fraction=0.046)
        
        # C. Difference (particles + noise only)
        im = axes[i, 2].imshow(diff, cmap='hot')
        axes[i, 2].set_title(f'Difference (particles + noise, max: {diff.max():.1f})')
        fig.colorbar(im, ax=axes[i, 2], fraction=0.046)
        
        # Mark true centers on all panels
        ys, xs = np.where(labels[2] > 0)
        for ax in axes[i]:
            ax.scatter(xs, ys, color='lime', marker='+', s=100, linewidths=2)
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## Construct single frame DECODE model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions


class conv_block(nn.Module):
    def __init__(self, in_channels, out_channels, dilation=1):
        super().__init__()
        # bias=False because both convs are immediately followed by BatchNorm,
        # whose beta absorbs the effective bias.
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1,
                      padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                      padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ELU(),
        )

    def forward(self, x):
        return self.conv(x)


class up_conv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # No norm here — bias stays True.
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1,
                      padding='same', bias=True),
        )

    def forward(self, x):
        return self.up(x)


class multi_head(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # First conv is followed by BatchNorm -> bias=False. Final 1x1 conv has
        # no norm after it and keeps bias=True (the p-head bias init of -8.1
        # depends on it).
        self.multi_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1,
                      padding='same', bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ELU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1,
                      padding='same', bias=True),
        )

    def forward(self, x):
        return self.multi_head(x)


class GaussianMixtureModel(nn.Module):
    def __init__(self, num_channels) -> None:
        super().__init__()
        self.num_channels = num_channels

        # =====================================================================
        # ENCODER — 3 levels (36 -> 72 -> 144) + dilated bottleneck (288)
        # Resolution for 128^2 input: 128 -> 64 -> 32 -> 16 (bottleneck)
        # =====================================================================
        self.Conv1 = conv_block(num_channels, 36)
        self.Maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.Conv2 = conv_block(36, 72)
        self.Maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.Conv3 = conv_block(72, 144)
        self.Maxpool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck at 16x16 (128^2 input) — dilations (2, 4).
        # Feature-map spans of 5 and 9 px, mapping to ~40 and ~72 input pixels
        # — 4-7x the PSF width, healthy without over-integrating.
        self.Conv4 = nn.Sequential(
            conv_block(144, 288, dilation=2),
            conv_block(288, 288, dilation=4),
        )

        # =====================================================================
        # DECODER — 3 upsampling levels back to full resolution (128x128)
        # =====================================================================
        self.Up3 = up_conv(288, 144)
        self.Up_conv3 = conv_block(288, 144)   # 144 skip + 144 up = 288

        self.Up2 = up_conv(144, 72)
        self.Up_conv2 = conv_block(144, 72)    # 72 skip + 72 up = 144

        self.Up1 = up_conv(72, 36)
        self.Up_conv1 = conv_block(72, 36)     # 36 skip + 36 up = 72

        # Scale on the tanh mu_xy offset. Fixed at 6.0 for training and
        # inference. Plain attribute (not in the state_dict).
        self.tanh_scale = 6.0

        # =====================================================================
        # HEADS — p (1), mu_xy (2), mu_a (1), sigma (3), bg (1)
        # =====================================================================
        self.head_p     = multi_head(36, 1)
        self.head_mu_xy = multi_head(36, 2)
        self.head_mu_a  = multi_head(36, 1)
        self.head_sigma = multi_head(36, 3)
        self.head_bg    = multi_head(36, 1)

        self.initialize_weights()

        # =================================================================
        # p-head init: bias -8.1 -> sigmoid(-8.1) ~ 0.0003, weights zero.
        # Runs AFTER initialize_weights so it's not overwritten.
        # =================================================================
        nn.init.constant_(self.head_p.multi_head[-1].bias, -8.1)
        nn.init.zeros_(self.head_p.multi_head[-1].weight)

    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward_heads(self, x):
        """Encoder+decoder+heads, activations applied, no scalar scaling.
        Returned tensors:  p (B,1,H,W), mu_xy (B,2,H,W), mu_a (B,1,H,W),
                           sigma (B,3,H,W), bg (B,1,H,W)
        """
        # --- Encoder ---
        x1 = self.Conv1(x)                          # (B, 36, 128, 128)
        x2 = self.Conv2(self.Maxpool1(x1))          # (B, 72,  64,  64)
        x3 = self.Conv3(self.Maxpool2(x2))          # (B, 144, 32,  32)
        x4 = self.Conv4(self.Maxpool3(x3))          # (B, 288, 16,  16)

        # --- Decoder ---
        d3 = self.Up3(x4)                            # (B, 144, 32, 32)
        d3 = torch.cat((x3, d3), dim=1)              # (B, 288, 32, 32)
        d3 = self.Up_conv3(d3)                       # (B, 144, 32, 32)

        d2 = self.Up2(d3)                            # (B, 72, 64, 64)
        d2 = torch.cat((x2, d2), dim=1)              # (B, 144, 64, 64)
        d2 = self.Up_conv2(d2)                       # (B, 72, 64, 64)

        d1 = self.Up1(d2)                            # (B, 36, 128, 128)
        d1 = torch.cat((x1, d1), dim=1)              # (B, 72, 128, 128)
        d1 = self.Up_conv1(d1)                       # (B, 36, 128, 128)

        # --- Heads (activations only; scalars applied outside) ---
        p     = torch.sigmoid(self.head_p(d1))       # (B, 1, H, W)
        mu_xy = torch.tanh(self.head_mu_xy(d1))      # (B, 2, H, W)
        mu_a  = torch.sigmoid(self.head_mu_a(d1))    # (B, 1, H, W)
        sigma = torch.sigmoid(self.head_sigma(d1))   # (B, 3, H, W)
        bg    = self.head_bg(d1)                     # (B, 1, H, W), linear
        return p, mu_xy, mu_a, sigma, bg

    def forward(self, x, training=False):
        """Returns (p, pxyn_mean, pxy_std, bg) as used by the loss/eval code.
        pxyn_mean = [mu_xy * tanh_scale, mu_a * 100];  pxy_std in (0.1, 2.5).
        """
        p, mu_xy, mu_a, sigma, bg = self.forward_heads(x)
        pxyn_mean = torch.cat([mu_xy * self.tanh_scale, mu_a * 100.0], dim=1)  # (B, 3, H, W)
        pxy_std   = sigma * 2.4 + 0.1                                          # sigma in (0.1, 2.5)
        return p, pxyn_mean, pxy_std, bg

    def forward_export(self, x):
        """Inference-only path for ONNX export: (p, mu_xy).
        mu_a, sigma, bg are training-only; tanh_scale is applied host-side.
        """
        p, mu_xy, _, _, _ = self.forward_heads(x)
        return p, mu_xy


# =========================================================================
# HELPER: get_ground_truth_targets
# =========================================================================
def get_ground_truth_targets(labels, xy_coord):
    batch_size = labels.shape[0]
    targets_list = []
    for b in range(batch_size):
        presence_map = labels[b, 2, ...] > 0
        indices = torch.nonzero(presence_map, as_tuple=False)
        if len(indices) == 0:
            targets_list.append(None)
            continue
        y_idx, x_idx = indices[:, 0], indices[:, 1]
        vals = labels[b, :, y_idx, x_idx].t()
        grid_x = xy_coord[0, 0, y_idx, x_idx]
        grid_y = xy_coord[0, 1, y_idx, x_idx]
        vals[:, 0] = vals[:, 0] + grid_x
        vals[:, 1] = vals[:, 1] + grid_y
        targets_list.append(vals)
    return targets_list


# =========================================================================
# LOSS — GMMLoss
# =========================================================================
class GMMLoss(nn.Module):
    def __init__(self, img_size, device) -> None:
        super(GMMLoss, self).__init__()
        x_coord = torch.linspace(start=0.0, end=img_size, steps=img_size + 1)[:-1].unsqueeze(0) + 0.5
        y_coord = torch.linspace(start=0.0, end=img_size, steps=img_size + 1)[:-1].unsqueeze(1) + 0.5
        self._xy_coord = torch.cat((
            x_coord.expand(1, 1, img_size, img_size),
            y_coord.expand(1, 1, img_size, img_size)
        ), 1).to(device=device)

    def forward(self, p, pxyn_mean, pxy_std, bg_pred, bg_target, labels):
        batch_size = p.shape[0]

        # ==============================================================
        # 1. COUNT LOSS — Bernoulli-Gaussian log-prob.
        #    Treats each pixel as Bernoulli(p); total count ≈ Normal(Σp, Σp(1-p)).
        #    The variance term Σp(1-p) is maximized at p=0.5 and zero at p∈{0,1},
        #    so minimizing -log_prob also drives p toward binary values.
        # ==============================================================
        p_sum = p.sum(dim=(-2, -1)).squeeze(-1)                                  # (B,)
        p_var = (p - p ** 2).sum(dim=(-2, -1)).squeeze(-1).clamp(min=1e-6)        # (B,)
        labels_intensity = labels[:, 2, ...] > 0
        gt_count = labels_intensity.sum(dim=(-2, -1)).float()                     # (B,)
        count_dist = torch.distributions.Normal(p_sum, p_var.sqrt())
        count_loss = -count_dist.log_prob(gt_count).mean()

        # ==============================================================
        # 2. GMM LOSS — full mixture (DECODE-style). Categorical over all
        #    pixels with Σp_all normalization. Per-pixel µ/σ contributions
        #    via MixtureSameFamily; the implicit bg suppression from the
        #    denominator (1/Σp_all per pixel) is preserved naturally.
        # ==============================================================
        gt_targets_list = get_ground_truth_targets(labels, self._xy_coord)
        total_gmm_log_prob = 0
        valid_batches = 0

        p_flat = p.view(batch_size, -1)
        p_normed = p_flat / (p_flat.sum(dim=1, keepdim=True) + 1e-6)

        pxyn_mean_abs = pxyn_mean.clone()
        pxyn_mean_abs[:, 0, ...] = pxyn_mean_abs[:, 0, ...] + self._xy_coord[:, 0, ...]
        pxyn_mean_abs[:, 1, ...] = pxyn_mean_abs[:, 1, ...] + self._xy_coord[:, 1, ...]
        mu_flat  = pxyn_mean_abs.view(batch_size, 3, -1).permute(0, 2, 1)
        std_flat = pxy_std.view(batch_size, 3, -1).permute(0, 2, 1)

        for b in range(batch_size):
            targets = gt_targets_list[b]
            if targets is None or targets.shape[0] == 0:
                continue
            valid_batches += 1
            mix  = torch.distributions.Categorical(probs=p_normed[b] + 1e-8)
            comp = torch.distributions.Independent(
                       torch.distributions.Normal(mu_flat[b], std_flat[b] + 1e-6), 1)
            gmm  = torch.distributions.mixture_same_family.MixtureSameFamily(mix, comp)
            log_probs = gmm.log_prob(targets)
            total_gmm_log_prob += log_probs.sum()

        gmm_loss = -(total_gmm_log_prob / valid_batches) if valid_batches > 0 \
                   else torch.tensor(0.0, device=p.device)

        # ---- Diagnostics (logged only, not part of the loss) ----
        # p-weighted mean std at firing pixels (lower = tighter Gaussians).
        with torch.no_grad():
            std_per_pixel = pxy_std[:, :2, ...].mean(dim=1, keepdim=True)   # (B, 1, H, W)
            self.last_p_weighted_std = ((std_per_pixel * p).sum() / (p.sum() + 1e-6)).item()
            # Entropy of the normalized p map (lower = more concentrated).
            p_entropy = -(p_normed * torch.log(p_normed + 1e-8)).sum(dim=1)
            self.last_p_entropy = p_entropy.mean().item()

        # ==============================================================
        # 3. BG LOSS — MSE between predicted bg and the normalized clean bg.
        #    Forces the shared backbone to disentangle bg structure from
        #    particle signal; the p-head then reads cleaner features.
        # ==============================================================
        bg_loss = F.mse_loss(bg_pred, bg_target)

        return count_loss, gmm_loss, bg_loss

## Construct train function

In [ ]:
def get_loss_weights(epoch):
    """Phased loss weights. bg stays at 1000 throughout.
    Phase 1 (0-300):    bg dominant, others at 1/10 normal — bg head pretraining
    Phase 2 (300-600):  others grow linearly back to normal
    Phase 3+ (600+):    bg=1000 stable, others at standard values
    """
    if epoch < 300:
        return {'count': 1.0, 'gmm': 0.1, 'bg': 1000.0}
    elif epoch < 600:
        prog = (epoch - 300) / 300
        return {
            'count': 1.0 + (10.0 - 1.0) * prog,
            'gmm':   0.1 + (1.0 - 0.1) * prog,
            'bg':    1000.0,
        }
    else:
        return {'count': 10.0, 'gmm': 1.0, 'bg': 1000.0}


def visualize_result(epoch, GT, images, p, pxy_mean, pxy_std,
                     true_counts_list, predicted_counts_p_list,
                     save_image=False, output_dir=None, eval=False):
    fig_key = ['IMG',
               'LABEL_GT',
               'P_COUNT',
               'X_MEAN',
               'Y_MEAN',
               'INTENSI',
               'X_VAR',
               'Y_VAR',
               'I_VAR',
               ]

    fig_dict = {'IMG'      : images[-1].detach().cpu(),
                'LABEL_GT' : GT[-1, ...].detach().cpu(),
                'P_COUNT'  : p[-1].detach().cpu().numpy(),
                'X_MEAN'   : pxy_mean[-1, 0, ...].detach().cpu().numpy(),
                'Y_MEAN'   : pxy_mean[-1, 1, ...].detach().cpu().numpy(),
                'INTENSI'  : pxy_mean[-1, 2, ...].detach().cpu().numpy(),
                'X_VAR'    : pxy_std[-1, 0, ...].detach().cpu().numpy(),
                'Y_VAR'    : pxy_std[-1, 1, ...].detach().cpu().numpy(),
                'I_VAR'    : pxy_std[-1, 2, ...].detach().cpu().numpy(),
                }

    if not epoch % 10:
        px = 1 / plt.rcParams['figure.dpi']
        fig, axs = plt.subplots(2, 5, figsize=(2500 * px, 1000 * px))

        for i, ax in enumerate(axs.flatten()):
            if i == 9:
                if len(true_counts_list) > 0:
                    max_count = max(max(true_counts_list), max(predicted_counts_p_list)) + 1
                    true_line = [[0, max_count]] * 2
                    ax.plot(*true_line, 'r-', alpha=0.5)
                    ax.scatter(true_counts_list, predicted_counts_p_list, s=10, c="blue", alpha=0.5)
                ax.set_title("Counting Accuracy")
                ax.set_xlabel("True Counts")
                ax.set_ylabel("Predicted Counts")
                continue

            IMG = np.squeeze(fig_dict[fig_key[i]])
            a = ax.imshow(IMG, cmap='viridis' if 'VAR' in fig_key[i] else 'gray')
            ax.set_title(fig_key[i])
            fig.colorbar(a, ax=ax)

        if save_image and output_dir:
            mode = 'eval' if eval else 'train'
            save_path = f'{output_dir}/{epoch}_epoch_{mode}.png'
            plt.savefig(save_path, dpi=300)

        if not epoch % 50:
            plt.show()
        else:
            plt.close()


def evaluate(model, loss_fun, data_loader, device, epoch, output_dir):
    model.eval()
    loss = 0
    true_counts_list, predicted_counts_p_list = [], []

    with torch.no_grad():
        for images, labels, bg_target in data_loader:
            images    = images.to(device=device)
            labels    = labels.to(device=device)
            bg_target = bg_target.to(device=device)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                p, pxy_mean, pxy_std, bg_pred = model(images, training=False)

            # Cast bf16 model outputs back to fp32 for stable loss computation
            p        = p.float()
            pxy_mean = pxy_mean.float()
            pxy_std  = pxy_std.float()
            bg_pred  = bg_pred.float()

            # ---- loss components: count, gmm, bg ----
            count_loss, gmm_loss, bg_loss = loss_fun(
                p, pxy_mean, pxy_std, bg_pred, bg_target, labels)
            _w = get_loss_weights(epoch)
            loss += (_w['count'] * count_loss + _w['gmm'] * gmm_loss + _w['bg'] * bg_loss).mean()

            for GT_single, p_single in zip(labels[:, -1, ...] > 0, p):
                true_counts = torch.sum(GT_single).item()
                predicted_counts_p = torch.sum(p_single).item()
                true_counts_list.append(true_counts)
                predicted_counts_p_list.append(predicted_counts_p)

        GT_batch = labels[:, -1, ...] > 0
        visualize_result(epoch, GT_batch, images, p, pxy_mean, pxy_std,
                         true_counts_list, predicted_counts_p_list,
                         save_image=True, output_dir=output_dir, eval=True)

    return loss


def _unwrapped_state_dict(model):
    """Return a state_dict whose keys match the original (uncompiled) model.
    torch.compile wraps the model and prefixes keys with "_orig_mod.".
    Stripping that prefix keeps saved checkpoints loadable into a fresh
    (non-compiled) GaussianMixtureModel at inference time."""
    if hasattr(model, '_orig_mod'):
        return model._orig_mod.state_dict()
    return model.state_dict()


def train(model, num_epoch, start_epoch, optimizer, scheduler, loss_fun,
          device, writer, output_dir, bg_tiles, window_size,
          bg_tiles_eval=None):
    # bg_tiles_eval is the held-out FOV pool; falls back to bg_tiles if
    # not supplied.
    if bg_tiles_eval is None:
        bg_tiles_eval = bg_tiles
    model.to(device=device)

    train_loss = torch.zeros(num_epoch)
    test_loss  = torch.zeros(num_epoch)

    best_loss = np.inf

    # --- BATCH SIZE --- (effective batch = 16 * 4 = 64)
    actual_dataloader_batch_size = 16
    accumulation_steps = 4

    num_samples_train = 600
    num_samples_eval  = 200

    print(f"Training Config: Image {window_size}, "
          f"Effective Batch {actual_dataloader_batch_size * accumulation_steps}")

    pbar = tqdm(range(start_epoch, num_epoch), total=num_epoch, initial=start_epoch)

    for epoch in pbar:
        epoch_start_time = time.time()
        model.train()

        # ---- per-epoch loss weights ----
        _w  = get_loss_weights(epoch)

        batch_loss_list = []
        count_loss_list, gmm_loss_list, bg_loss_list = [], [], []
        p_entropy_list, std_pen_list = [], []
        true_counts_list, predicted_counts_p_list = [], []

        train_dataset = GaussianDataset(num_batches=num_samples_train,
                                         bg_tiles=bg_tiles, img_size=window_size)
        train_dataloader = DataLoader(train_dataset,
                                       batch_size=actual_dataloader_batch_size,
                                       shuffle=True, num_workers=0, pin_memory=True)

        optimizer.zero_grad()

        for i, (images, labels, bg_target) in enumerate(train_dataloader):
            images    = images.to(device=device)
            labels    = labels.to(device=device)
            bg_target = bg_target.to(device=device)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                p, pxy_mean, pxy_std, bg_pred = model(images, training=True)

            # Cast bf16 model outputs back to fp32 for stable loss computation
            p        = p.float()
            pxy_mean = pxy_mean.float()
            pxy_std  = pxy_std.float()
            bg_pred  = bg_pred.float()

            # ---- loss components: count, gmm, bg ----
            count_loss, gmm_loss, bg_loss = loss_fun(
                p, pxy_mean, pxy_std, bg_pred, bg_target, labels)

            # Weighted sum (weights from get_loss_weights). bg_loss is auxiliary:
            # it supervises the shared backbone to separate bg from particles.
            loss = (_w['count'] * count_loss) + (_w['gmm'] * gmm_loss) + (_w['bg'] * bg_loss)

            loss_item = loss.item()

            loss = loss / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.3)
                optimizer.step()
                optimizer.zero_grad()

            batch_loss_list.append(loss_item)
            count_loss_list.append(count_loss.item())
            gmm_loss_list.append(gmm_loss.item())
            bg_loss_list.append(bg_loss.item())
            p_entropy_list.append(loss_fun.last_p_entropy)
            std_pen_list.append(loss_fun.last_p_weighted_std)

            for GT_single, p_val in zip(labels[:, -1, ...] > 0, p):
                true_counts_list.append(torch.sum(GT_single).item())
                predicted_counts_p_list.append(torch.sum(p_val).item())

        if (len(train_dataloader) % accumulation_steps != 0):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.3)
            optimizer.step()
            optimizer.zero_grad()

        # Visualize
        GT_batch = labels[:, -1, ...] > 0
        visualize_result(epoch, GT_batch, images, p, pxy_mean, pxy_std,
                         true_counts_list, predicted_counts_p_list,
                         save_image=True, output_dir=output_dir, eval=False)

        scheduler.step()
        lr = scheduler.get_last_lr()[0]
        train_loss[epoch] = np.mean(batch_loss_list)

        # --- EVALUATION ---
        eval_dataset = GaussianDataset(num_batches=num_samples_eval,
                                        bg_tiles=bg_tiles_eval, img_size=window_size)
        eval_dataloader = DataLoader(eval_dataset,
                                      batch_size=actual_dataloader_batch_size,
                                      shuffle=True, num_workers=0, pin_memory=True)

        loss_test = evaluate(model=model, loss_fun=loss_fun,
                              data_loader=eval_dataloader, device=device,
                              epoch=epoch, output_dir=output_dir)
        test_loss[epoch] = loss_test.item()

        if writer:
            writer.add_scalar('Loss/Count',      np.mean(count_loss_list), epoch + 1)
            writer.add_scalar('Loss/GMM',        np.mean(gmm_loss_list),   epoch + 1)
            writer.add_scalar('Loss/BG',         np.mean(bg_loss_list),    epoch + 1)
            writer.add_scalar('Diag/PEntropy',     np.mean(p_entropy_list), epoch + 1)  # spread of p (lower = more concentrated)
            writer.add_scalar('Diag/PWeightedStd', np.mean(std_pen_list),   epoch + 1)  # p-weighted mean std (lower = tighter Gaussians)
            writer.add_scalar('Loss/Train',      train_loss[epoch],        epoch + 1)
            writer.add_scalar('Loss/Test',       test_loss[epoch],         epoch + 1)
            writer.add_scalar('LR',              lr,                       epoch + 1)
            writer.add_scalar('Weights/count',   _w['count'],              epoch + 1)
            writer.add_scalar('Weights/gmm',     _w['gmm'],                epoch + 1)
            writer.add_scalar('Weights/bg',      _w['bg'],                 epoch + 1)

        pbar.set_description(
            f"Epoch {epoch+1} | Loss: {np.mean(batch_loss_list):.4f} "
            f"| Cnt: {np.mean(count_loss_list):.1f} "
            f"| GMM: {np.mean(gmm_loss_list):.1f} "
            f"| BG: {np.mean(bg_loss_list):.3f} "
            f"| Test: {loss_test.item():.4f}")

        # --- SAVING LOGIC ---
        if loss_test < best_loss:
            best_loss = loss_test
            torch.save(_unwrapped_state_dict(model), f'{output_dir}/LOCA_PRAM_BEST.pth')

        if (epoch + 1) % 20 == 0:
            chk_name = f'{output_dir}/LOCA_PRAM_checkpoint_epoch_{epoch+1}.pth'
            torch.save(_unwrapped_state_dict(model), chk_name)

    return train_loss, test_loss, model

In [ ]:
import time
import torch # Assuming torch is needed inside generate_simulation_image
import numpy as np # Assuming numpy is needed inside generate_simulation_image
if __name__ == '__main__':
    # Test single image generation time
    num_test_images = 10 # Generate 100 images to get an average
    single_image_gen_times = []
    
    for _ in range(num_test_images):
        start_gen_time = time.time()
        # Call your generate_simulation_image function as it would be called by __getitem__
        # Use actual img_size and bg_tiles
        generated_image, _, _ = generate_simulation_image(
            img_size=window_size, # Use your actual window_size
            bg_tiles=bg_tiles,     # Use your actual bg_tiles
            num_gaus_low=0,
            num_gaus_high=40,
            labels_channel=3
        )
        end_gen_time = time.time()
        single_image_gen_times.append(end_gen_time - start_gen_time)
    
    average_gen_time = np.mean(single_image_gen_times)
    print(f"Average time to generate a single image: {average_gen_time:.4f} seconds")
    print(f"Estimated time to fill DataLoader queue (approx 2-3x num_workers * batch_size images):")
    # Assuming a queue size that needs about 2-3 times the number of workers times batch_size images
    # 2.5 * 15 workers * 10 images/batch * average_gen_time = total queue fill time
    estimated_queue_fill_time = (2.5 * 15 * 10 * average_gen_time) # Rough estimate for 15 workers, batch 10
    print(f"  Roughly {estimated_queue_fill_time:.2f} seconds based on 15 workers and batch size 10.")

## Main

In [ ]:
from torch.distributions import Distribution
import datetime
import gc
import os

if __name__ == '__main__':
    Distribution.set_default_validate_args(False)

    # ---- Ampere/A6000 free wins ----
    torch.backends.cudnn.benchmark = True            # pick fastest conv algo per shape
    torch.backends.cuda.matmul.allow_tf32 = True     # TF32 matmul on Ampere
    torch.backends.cudnn.allow_tf32 = True

    # --- HYPERPARAMETERS ---
    learning_rate = 3e-5
    momentum      = 0.9
    weight_decay  = 0.0
    gamma         = 0.1

    current_time = datetime.datetime.now().strftime('%Y.%m.%d_%H.%M.%S')
    output_dir   = 'runs/' + current_time + '/'

    batch_size = 16
    seed       = 2
    max_epoch  = 5000  # warmup(0-50) -> flat(50-2000) -> cosine(2000-5000)

    in_channel = 1
    linear_lr_scheduler_max = 50

    # window_size is set globally above based on DOWNSCALE_FACTOR

    torch.manual_seed(seed)

    gc.collect()
    torch.cuda.empty_cache()

    model  = GaussianMixtureModel(num_channels=in_channel)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f" Current device is: {device}")

    loss_fun = GMMLoss(img_size=window_size[0], device=device)
    model.to(device)

    # ---- torch.compile: 1.3-2x speedup on Ampere ----
    # The first epoch will be slow as PyTorch JIT-compiles kernels; subsequent
    # epochs are much faster. If compile fails on this hardware, comment out.
    #model = torch.compile(model)
    writer = SummaryWriter(output_dir, flush_secs=10)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.999),
        weight_decay=weight_decay,
    )

    import math
    def lr_lambda(epoch):
        # Warmup -> flat -> cosine -> flat tail.
        #   0-50:      linear 0 -> 1.0                    (0 -> 3e-5)
        #   50-2000:   flat at 1.0                        (3e-5)
        #   2000-5000: cosine 1.0 -> TAIL_MIN_FRAC        (3e-5 -> 1.5e-6)
        #   5000+:     flat at TAIL_MIN_FRAC
        WARMUP_END    = 50
        FLAT_END      = 2000
        COSINE_END    = 5000
        TAIL_MIN_FRAC = 0.05   # 1.5e-6 floor

        if epoch < WARMUP_END:
            return (epoch + 1) / WARMUP_END
        if epoch < FLAT_END:
            return 1.0
        if epoch < COSINE_END:
            tail_progress = (epoch - FLAT_END) / (COSINE_END - FLAT_END)
            return TAIL_MIN_FRAC + (1.0 - TAIL_MIN_FRAC) * 0.5 * (1 + math.cos(math.pi * tail_progress))
        return TAIL_MIN_FRAC

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    # Set to a LOCA_PRAM_checkpoint_epoch_<N>.pth path to resume; None trains from scratch.
    checkpoint_path = None

    start_epoch = 0

    if checkpoint_path and os.path.isfile(checkpoint_path):
        print(f"Loading checkpoint from: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint)

        try:
            filename = os.path.basename(checkpoint_path)
            raw_epoch_part = filename.split('_')[-1]
            epoch_str = raw_epoch_part.split('.')[0]
            start_epoch = int(epoch_str) + 1
            print(f"Resuming from epoch: {start_epoch}")
        except (IndexError, ValueError):
            print("Could not parse epoch number. Starting from 0.")
            start_epoch = 0
    else:
        print("Training from scratch.")

    if start_epoch > 0:
        print(f"⏩ Fast-forwarding scheduler to step {start_epoch}...")
        for _ in range(start_epoch):
            scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"✅ Current Learning Rate: {current_lr:.2e}")

    # Quick sanity check: verify p.sum() at initialization
    model.eval()
    with torch.no_grad():
        dummy = torch.randn(1, 1, window_size[0], window_size[1]).to(device)
        p_init, _, _, _ = model(dummy)
        print(f"✅ Initial p.sum() = {p_init.sum().item():.1f}  (target range: 0–7)")
    model.train()

    print("Starting training...")

    train_loss, test_loss, model = train(
        model=model,
        num_epoch=max_epoch,
        start_epoch=start_epoch,
        optimizer=optimizer,
        scheduler=scheduler,
        loss_fun=loss_fun,
        device=device,
        writer=writer,
        output_dir=output_dir,
        bg_tiles=bg_tiles_train,
        bg_tiles_eval=bg_tiles_eval,
        window_size=window_size
    )

    plt.plot(train_loss, 's-', label='Train')
    plt.plot(test_loss, 'o-', label='Test')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Model loss (final test loss: %.2f)' % test_loss[-1])
    plt.show()

## Clear cache

In [ ]:
import gc
import torch
if __name__ == '__main__':
    gc.collect()
    torch.cuda.empty_cache()

## Test the trained model

In [ ]:
###############################################################################
# VALIDATION TEST — synthetic tiles on held-out background FOVs
#
# Preprocessing + inference match training's evaluate():
#   - normalize with the CLEAN bg mask (bg_tile > MARKER_THRESHOLD)
#   - inference under torch.autocast(bfloat16), outputs cast back to fp32
#
# DETECTOR: NMS on p_map with μ-refined sub-pixel positions; detections in
# the forbidden mask (marker buffer + edge margin) are dropped.
###############################################################################
import glob
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2


def synthesize_test_tile(bg_tile, num_gaus_low, num_gaus_high):
    """Particle synthesis that returns amplitudes alongside positions."""
    H, W = bg_tile.shape
    image = bg_tile.copy().astype(np.float32)

    forbidden = compute_forbidden_mask(bg_tile)
    if MARGIN > 0:
        forbidden[:MARGIN, :] = True
        forbidden[-MARGIN:, :] = True
        forbidden[:, :MARGIN] = True
        forbidden[:, -MARGIN:] = True
    pys, pxs = np.where(~forbidden)
    active_frac = pys.size / forbidden.size

    nominal = np.random.randint(num_gaus_low, num_gaus_high)
    scaled = nominal * active_frac
    n_floor = int(scaled)
    n_place = n_floor + (1 if np.random.random() < (scaled - n_floor) else 0)
    if pys.size == 0:
        n_place = 0

    positions, amplitudes = [], []
    x_vec = np.arange(W); y_vec = np.arange(H)

    for _ in range(n_place):
        k = np.random.randint(0, pys.size)
        x_c = pxs[k] + np.random.uniform(0, 1)
        y_c = pys[k] + np.random.uniform(0, 1)
        amp = float(sample_amplitude())
        sx = max(SIGMA_MIN, np.random.normal(loc=SIGMA_X_LOC, scale=SIGMA_X_SCALE))
        sy = max(SIGMA_MIN, np.random.normal(loc=SIGMA_Y_LOC, scale=SIGMA_Y_SCALE))
        g_x = np.exp(-((x_vec - x_c) ** 2) / (2 * sx ** 2))
        g_y = np.exp(-((y_vec - y_c) ** 2) / (2 * sy ** 2))
        image -= np.outer(g_y, g_x) * amp
        positions.append((x_c, y_c))
        amplitudes.append(amp)

    image = np.clip(image, 0, 65535)
    return (image,
            np.array(positions) if positions else np.zeros((0, 2)),
            np.array(amplitudes))


def compute_forbidden_mask_fast(image, buffer_px=None, threshold=None, margin=None):
    """Forbidden = pixels within buffer_px of any marker pixel, OR within
    margin of tile edge. Same definition as training, faster impl."""
    if buffer_px is None: buffer_px = BUFFER_PX
    if threshold is None: threshold = MARKER_THRESHOLD
    if margin is None:    margin = MARGIN

    H, W = image.shape
    non_marker = (image > threshold).astype(np.uint8)

    if non_marker.all():
        forbidden = np.zeros((H, W), dtype=bool)
    else:
        dist = cv2.distanceTransform(non_marker, cv2.DIST_L2, 3)
        forbidden = dist <= buffer_px

    if margin > 0:
        forbidden[:margin, :]  = True
        forbidden[-margin:, :] = True
        forbidden[:, :margin]  = True
        forbidden[:, -margin:] = True
    return forbidden


def detect_via_nms(p_map_np, mu_map_np, forbidden_mask, p_threshold,
                   nms_kernel=5, refine_subpixel=True):
    above = (p_map_np > p_threshold) & ~forbidden_mask
    if not above.any():
        return np.array([]), np.array([])
    kernel = np.ones((nms_kernel, nms_kernel), dtype=np.uint8)
    p_max = cv2.dilate(p_map_np.astype(np.float32), kernel)
    peaks = above & (p_map_np >= p_max - 1e-9)
    yi, xi = np.where(peaks)
    if refine_subpixel:
        rx = xi.astype(np.float64) + mu_map_np[0, yi, xi]
        ry = yi.astype(np.float64) + mu_map_np[1, yi, xi]
    else:
        rx = xi.astype(np.float64); ry = yi.astype(np.float64)
    return rx, ry


def match_detections_fast(true_pos, pred_x, pred_y, match_radius=10):
    if len(true_pos) == 0:
        return 0, len(pred_x), 0, []
    if len(pred_x) == 0:
        return 0, 0, len(true_pos), []
    matched = set(); out = []
    for pi in range(len(pred_x)):
        d = np.hypot(true_pos[:, 0] - pred_x[pi], true_pos[:, 1] - pred_y[pi])
        for ti in np.argsort(d):
            if d[ti] > match_radius: break
            if ti not in matched:
                matched.add(ti); out.append((ti, pi, d[ti])); break
    tp = len(out)
    return tp, len(pred_x) - tp, len(true_pos) - tp, out


def run_validation_test(model, device, bg_tiles, num_test_tiles=200,
                        p_threshold=0.006, match_radius=10,
                        num_gaus_low=0, num_gaus_high=10,
                        noise_floor=2.7, nms_kernel=7,
                        p_threshold_grid=None, nms_kernel_grid=None,
                        show_examples=5,
                        threshold_sweep=True, amplitude_analysis=True):
    model.eval()
    np.random.seed(42)

    per_tile = []

    print(f"Running {num_test_tiles} tiles, num_gaus in [{num_gaus_low}, {num_gaus_high})...")
    print(f"Detector: NMS (nms_kernel={nms_kernel}), "
          f"suppression: marker-buffer ({BUFFER_PX}px) + edge ({MARGIN}px)")

    for ti in range(num_test_tiles):
        # 1. Generate
        bg_tile = bg_tiles[np.random.randint(len(bg_tiles))]
        image, true_pos, true_amp = synthesize_test_tile(
            bg_tile, num_gaus_low, num_gaus_high)

        # 2. Normalize — same as training's __getitem__ (clean-bg mask)
        valid = bg_tile > MARKER_THRESHOLD
        if valid.any():
            m, s = image[valid].mean(), image[valid].std()
        else:
            m, s = image.mean(), image.std()
        if s == 0: s = 1
        img_n = (image - m) / s

        # 3. Inference — bf16 autocast + fp32 recast, matches training evaluate()
        x = torch.from_numpy(img_n).float().unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                p, pxy_mean, pxy_std, bg_pred = model(x, training=False)
            p        = p.float()
            pxy_mean = pxy_mean.float()
        p_map_np  = p[0, 0].cpu().numpy()
        mu_map_np = pxy_mean[0, :2].cpu().numpy()

        # 4. Forbidden mask (from the raw input)
        forbidden = compute_forbidden_mask_fast(image)

        # 5. Detect + match
        cx, cy = detect_via_nms(p_map_np, mu_map_np, forbidden,
                                p_threshold, nms_kernel=nms_kernel)
        tp, fp, fn, matches = match_detections_fast(
            true_pos, cx, cy, match_radius=match_radius)

        per_tile.append({
            'tile_i': ti, 'image': image,
            'p_map': p_map_np, 'mu_map': mu_map_np,
            'forbidden': forbidden,
            'true_pos': true_pos, 'true_amp': true_amp,
            'pred_x': cx, 'pred_y': cy,
            'tp': tp, 'fp': fp, 'fn': fn, 'matches': matches,
        })

    # ---- aggregate ----
    TP = sum(r['tp'] for r in per_tile)
    FP = sum(r['fp'] for r in per_tile)
    FN = sum(r['fn'] for r in per_tile)
    P  = TP / (TP + FP) if TP + FP > 0 else 0
    R  = TP / (TP + FN) if TP + FN > 0 else 0
    F1 = 2 * P * R / (P + R) if P + R > 0 else 0
    all_d = [m[2] for r in per_tile for m in r['matches']]

    print(f"\n{'='*60}\n  AGGREGATE (thr={p_threshold}, radius={match_radius}px)\n{'='*60}")
    print(f"  Tiles: {num_test_tiles}   True: {sum(len(r['true_pos']) for r in per_tile)}   "
          f"Detected: {sum(len(r['pred_x']) for r in per_tile)}")
    print(f"  TP={TP}  FP={FP}  FN={FN}")
    print(f"  Precision={P:.3f}   Recall={R:.3f}   F1={F1:.3f}")
    if all_d:
        print(f"  Localization error  mean={np.mean(all_d):.2f}   median={np.median(all_d):.2f} px")

    # ---- amplitude analysis ----
    if amplitude_analysis:
        all_tp_amps, all_fn_amps = [], []
        for r in per_tile:
            if len(r['true_amp']) == 0: continue
            matched_ti = set(m[0] for m in r['matches'])
            for ti2, amp in enumerate(r['true_amp']):
                (all_tp_amps if ti2 in matched_ti else all_fn_amps).append(amp)
        all_tp_amps = np.array(all_tp_amps)
        all_fn_amps = np.array(all_fn_amps)

        print(f"\n{'='*60}\n  GT AMPLITUDE: TP vs FN (noise floor ≈ {noise_floor})\n{'='*60}")
        if len(all_tp_amps) > 0:
            print(f"  TP (n={len(all_tp_amps)}): mean={all_tp_amps.mean():.2f}  "
                  f"median={np.median(all_tp_amps):.2f}  "
                  f"min={all_tp_amps.min():.2f}  max={all_tp_amps.max():.2f}")
        if len(all_fn_amps) > 0:
            print(f"  FN (n={len(all_fn_amps)}): mean={all_fn_amps.mean():.2f}  "
                  f"median={np.median(all_fn_amps):.2f}  "
                  f"min={all_fn_amps.min():.2f}  max={all_fn_amps.max():.2f}")
            for k in [2, 3, 5]:
                below = (all_fn_amps < k * noise_floor).sum()
                print(f"  FNs below {k}× noise (amp < {k*noise_floor:.1f}): "
                      f"{below}/{len(all_fn_amps)} ({below/len(all_fn_amps)*100:.1f}%)")

        if len(all_tp_amps) + len(all_fn_amps) > 0:
            all_amps = np.concatenate([all_tp_amps, all_fn_amps])
            all_hit  = np.concatenate([np.ones(len(all_tp_amps)),
                                       np.zeros(len(all_fn_amps))])
            findable = all_amps >= 3 * noise_floor
            if findable.sum() > 0:
                rec_findable = all_hit[findable].mean()
                print(f"\n  Recall on findable (amp ≥ 3× noise = {3*noise_floor:.1f}): "
                      f"{rec_findable:.3f}  "
                      f"({int(all_hit[findable].sum())}/{int(findable.sum())})")

        fig, ax = plt.subplots(1, 2, figsize=(14, 5))
        max_amp = max(
            all_tp_amps.max() if len(all_tp_amps) > 0 else 60,
            all_fn_amps.max() if len(all_fn_amps) > 0 else 60)
        bins = np.linspace(0, max_amp + 5, 30)

        ax[0].hist(all_tp_amps, bins=bins, alpha=0.6, color='seagreen',
                   label=f'TP (n={len(all_tp_amps)})', edgecolor='black')
        if len(all_fn_amps) > 0:
            ax[0].hist(all_fn_amps, bins=bins, alpha=0.75, color='crimson',
                       label=f'FN (n={len(all_fn_amps)})', edgecolor='black')
        for k, color, ls in [(1, 'gray', ':'), (3, 'darkorange', '--'),
                             (5, 'goldenrod', '--')]:
            ax[0].axvline(k * noise_floor, ls=ls, color=color, alpha=0.7,
                          label=f'{k}× noise ({k*noise_floor:.1f})')
        ax[0].set_xlabel('GT amplitude'); ax[0].set_ylabel('Count')
        ax[0].set_title('GT amplitude distribution by detection outcome')
        ax[0].legend(fontsize=9, loc='upper right'); ax[0].grid(alpha=0.3)

        if len(all_fn_amps) > 0 and len(all_tp_amps) > 0:
            bin_edges = np.linspace(0, max_amp + 5, 13)
            centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            width = bin_edges[1] - bin_edges[0]
            rates, counts = [], []
            for i in range(len(centers)):
                mm = (all_amps >= bin_edges[i]) & (all_amps < bin_edges[i+1])
                cnt = mm.sum()
                rates.append(all_hit[mm].mean() if cnt > 0 else 0)
                counts.append(cnt)
            ax[1].bar(centers, rates, width=width*0.9,
                      color='steelblue', edgecolor='black')
            for c, cnt in zip(centers, counts):
                if cnt > 0:
                    ax[1].text(c, 0.02, f'n={cnt}',
                               ha='center', va='bottom', fontsize=7)
            for k, color in [(3, 'darkorange'), (5, 'goldenrod')]:
                ax[1].axvline(k * noise_floor, ls='--', color=color, alpha=0.7)
            ax[1].set_xlabel('GT amplitude bin'); ax[1].set_ylabel('Recall')
            ax[1].set_title('Recall by amplitude')
            ax[1].set_ylim([0, 1.05]); ax[1].grid(alpha=0.3)
        plt.tight_layout(); plt.show()

    # ---- threshold x NMS-kernel sweep ----
    if threshold_sweep:
        thrs = list(p_threshold_grid) if p_threshold_grid is not None else \
               [0.001, 0.002, 0.003, 0.004, 0.006, 0.008, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.25]
        nks  = list(nms_kernel_grid)  if nms_kernel_grid  is not None else [3, 5, 7]
        print(f"\n{'='*64}\n  SWEEP (NMS, mask on)  thrs={len(thrs)} × nks={len(nks)}\n{'='*64}")
        f1_grid = np.zeros((len(nks), len(thrs)))
        p_grid  = np.zeros_like(f1_grid)
        r_grid  = np.zeros_like(f1_grid)
        for ii, nk in enumerate(nks):
            for jj, thr in enumerate(thrs):
                tp_s = fp_s = fn_s = 0
                for r in per_tile:
                    cx, cy = detect_via_nms(r['p_map'], r['mu_map'], r['forbidden'],
                                            thr, nms_kernel=nk)
                    t, f_, n_, _ = match_detections_fast(r['true_pos'], cx, cy, match_radius)
                    tp_s += t; fp_s += f_; fn_s += n_
                p_s = tp_s / (tp_s + fp_s) if tp_s + fp_s > 0 else 0
                r_s = tp_s / (tp_s + fn_s) if tp_s + fn_s > 0 else 0
                f1_s = 2 * p_s * r_s / (p_s + r_s) if p_s + r_s > 0 else 0
                f1_grid[ii, jj] = f1_s; p_grid[ii, jj] = p_s; r_grid[ii, jj] = r_s

        fig, axes = plt.subplots(1, len(nks), figsize=(6 * len(nks), 4.5),
                                 sharey=True)
        if len(nks) == 1: axes = [axes]
        for ax, ii, nk in zip(axes, range(len(nks)), nks):
            ax.plot(thrs, p_grid[ii], 'b-o', label='Precision')
            ax.plot(thrs, r_grid[ii], 'g-o', label='Recall')
            ax.plot(thrs, f1_grid[ii], 'r-o', lw=2, label='F1')
            ax.set_xscale('log'); ax.set_xlabel('p_threshold'); ax.set_ylabel('score')
            ax.set_title(f'nms_kernel={nk}'); ax.legend(); ax.grid(alpha=0.3)
            ax.set_ylim(0, 1.02)
        plt.tight_layout(); plt.show()
        bi, bj = np.unravel_index(np.argmax(f1_grid), f1_grid.shape)
        print(f"\n  Best F1={f1_grid[bi, bj]:.3f} at p_thr={thrs[bj]} nms_kernel={nks[bi]}  "
              f"P={p_grid[bi, bj]:.3f} R={r_grid[bi, bj]:.3f}")

    # ---- example tiles ----
    if show_examples > 0:
        ex = sorted(per_tile, key=lambda r: len(r['true_pos']), reverse=True)[:show_examples]
        fig, axes = plt.subplots(len(ex), 3, figsize=(15, 5 * len(ex)))
        if len(ex) == 1: axes = axes[np.newaxis, :]
        for row, r in enumerate(ex):
            axes[row, 0].imshow(r['image'], cmap='gray', interpolation='none')
            axes[row, 0].imshow(r['forbidden'], cmap='Reds', alpha=0.25,
                                interpolation='none')
            if len(r['true_pos']) > 0:
                axes[row, 0].scatter(r['true_pos'][:, 0], r['true_pos'][:, 1],
                                     facecolors='none', edgecolors='lime', s=80, lw=1.5,
                                     label=f"True ({len(r['true_pos'])})")
            if len(r['pred_x']) > 0:
                axes[row, 0].scatter(r['pred_x'], r['pred_y'], color='red', marker='x',
                                     s=80, lw=1.5, label=f"Pred ({len(r['pred_x'])})")
            axes[row, 0].legend(fontsize=9)
            axes[row, 0].set_title(f"Tile {r['tile_i']}  TP={r['tp']} FP={r['fp']} FN={r['fn']}  "
                                   f"(red=forbidden)")
            axes[row, 0].axis('off')

            im = axes[row, 1].imshow(r['p_map'], cmap='hot', interpolation='none')
            if len(r['pred_x']) > 0:
                axes[row, 1].scatter(r['pred_x'], r['pred_y'], color='cyan', marker='x',
                                     s=80, lw=1.5)
            axes[row, 1].set_title(f"P-map (max={r['p_map'].max():.3f})")
            axes[row, 1].axis('off')
            fig.colorbar(im, ax=axes[row, 1], fraction=0.046)

            mask = (r['p_map'] > p_threshold) & ~r['forbidden']
            axes[row, 2].imshow(r['image'], cmap='gray', alpha=0.5, interpolation='none')
            axes[row, 2].imshow(mask, cmap='Blues', alpha=0.5, interpolation='none')
            axes[row, 2].imshow(r['forbidden'], cmap='Reds', alpha=0.2,
                                interpolation='none')
            if len(r['pred_x']) > 0:
                axes[row, 2].scatter(r['pred_x'], r['pred_y'], color='red', marker='x',
                                     s=80, lw=1.5)
            if len(r['true_pos']) > 0:
                axes[row, 2].scatter(r['true_pos'][:, 0], r['true_pos'][:, 1],
                                     facecolors='none', edgecolors='lime', s=80, lw=1.5)
            axes[row, 2].set_title('detection mask (blue) + forbidden (red)')
            axes[row, 2].axis('off')
        plt.tight_layout(); plt.show()

    return per_tile


# ============ RUN ============
if __name__ == '__main__':
    # None -> LOCA_PRAM_BEST.pth from the most recent run in runs/.
    MODEL_PATH = None
    if MODEL_PATH is None:
        _runs = sorted(glob.glob(os.path.join('runs', '*', 'LOCA_PRAM_BEST.pth')))
        assert _runs, "no runs/*/LOCA_PRAM_BEST.pth found — train first or set MODEL_PATH"
        MODEL_PATH = _runs[-1]

    model = GaussianMixtureModel(num_channels=1)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Loading model: {MODEL_PATH}")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device)
    print(f"Device: {device} | bg_tiles_eval: {len(bg_tiles_eval)} tiles (held-out FOVs)")

    results = run_validation_test(
        model=model, device=device,
        bg_tiles=bg_tiles_eval,
        num_test_tiles=200,
        p_threshold=0.006,
        match_radius=10,
        num_gaus_low=0, num_gaus_high=10,
        noise_floor=2.7,
        nms_kernel=7,
        show_examples=5,
        threshold_sweep=True,
        amplitude_analysis=True,
    )